In [4]:
!pip install -q -U google-genai python-dotenv
!python.exe -m pip install --upgrade pip


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2


In [5]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")

client = genai.Client(api_key=api_key)

print("Gemini API client initialized successfully!")

Gemini API client initialized successfully!


In [9]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello there!


# Task 2: Tool Calling Fundamentals

In this task, Gemini's function calling capability is used to connect the language model with custom Python tools. Two simple tools are implemented: a calculator and a weather lookup stub. The model will decide which tool is appropriate, return structured arguments, and the Python program will execute the selected tool.


In [15]:
def calculator(a, b, operation):
    if operation == "add":
        return a + b
    
    elif operation == "subtract":
        return a - b
    
    elif operation == "multiply":
        return a * b
    
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    
    else:
        raise ValueError(f"Unknown operation: {operation}")
print(calculator(10, 5, "add"))
print(calculator(10, 5, "multiply"))

15
50


In [17]:
calculator_tool = {
    "type": "function",
    "name": "calculator",
    "description": "Performs basic arithmetic operations on two numbers.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {
                "type": "number",
                "description": "The first number."
            },
            "b": {
                "type": "number",
                "description": "The second number."
            },
            "operation": {
                "type": "string",
                "enum": [
                    "add",
                    "subtract",
                    "multiply",
                    "divide"
                ],
                "description": "The arithmetic operation to perform."
            }
        },
        "required": ["a", "b", "operation"]
    }
}

In [18]:
def get_weather(city):
    weather_data = {
        "karachi": {
            "temperature": 34,
            "condition": "Sunny"
        },
        "lahore": {
            "temperature": 31,
            "condition": "Partly Cloudy"
        },
        "islamabad": {
            "temperature": 27,
            "condition": "Cloudy"
        },
        "dubai": {
            "temperature": 38,
            "condition": "Sunny"
        }
    }

    city_key = city.strip().lower()

    if city_key not in weather_data:
        raise ValueError(
            f"Weather data is not available for {city}."
        )

    return weather_data[city_key]

In [19]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Returns simulated current weather information for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to look up."
            }
        },
        "required": ["city"]
    }
}

In [21]:
available_tools = {
    "calculator": calculator,
    "get_weather": get_weather
}

## Why Tool Descriptions Matter

Tool descriptions help the model understand what each tool does, when it should be used, and what information should be provided as input. Clear descriptions and well-defined parameter schemas improve tool selection and reduce incorrect or incomplete arguments.
## For example, 
if a tool is described as "Performs basic arithmetic operations on two numbers," the model can identify it as the appropriate tool for a calculation request. Similarly, describing the weather tool as a city-based weather lookup helps the model provide the correct city argument.


In [26]:
from google import genai
from google.genai import types

tools = types.Tool(
    function_declarations=[
        {
            "name": "calculator",
            "description": "Performs basic arithmetic operations on two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "The first number."
                    },
                    "b": {
                        "type": "number",
                        "description": "The second number."
                    },
                    "operation": {
                        "type": "string",
                        "enum": [
                            "add",
                            "subtract",
                            "multiply",
                            "divide"
                        ],
                        "description": "The arithmetic operation to perform."
                    }
                },
                "required": ["a", "b", "operation"]
            }
        },
        {
            "name": "get_weather",
            "description": "Returns simulated current weather information for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to look up."
                    }
                },
                "required": ["city"]
            }
        }
    ]
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is 25 multiplied by 8?",
    config=types.GenerateContentConfig(
        tools=[tools]
    )
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'a': 25,
            'b': 8,
            'operation': 'multiply'
          },
          name='calculator'
        ),
        thought_signature=b"\n\xef\x02\x01\x11M2\x0fu\rM\xb7[\xf0};\x13\x17v\xdc\n\x97\x1e\x95\xc3H\x04\xc6\xd3\x8d\xcb\xd9\xfb\xbd@[\xcb\x9d\xb4Rg'v\xd3\x19E\x0f\xf2f\xcfp\xfb9\xf4\xc0\x18\xc5.$\xf6\xaf\xb9\x1c\xdf\x97b\x8f\x1a\x0f\x8dmt\xf0\x8b[\x89\rCK\xc8\xee\xc1\x96\x85\xf8\xd5g\xbf\xac^\x86\xb6'\x90^'U...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='zNmeauqGOqSekdUPqKOQoQM' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=24,
  prompt_token_count=144,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<M

In [27]:
function_call = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        function_call = part.function_call
        break

if function_call:
    print("Function Name:", function_call.name)
    print("Arguments:", dict(function_call.args))
else:
    print("No function call was returned.")

Function Name: calculator
Arguments: {'b': 8, 'operation': 'multiply', 'a': 25}


In [28]:
if function_call:
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    result = available_tools[tool_name](**tool_args)

    print("Tool Result:", result)

Tool Result: 200


In [38]:
from google.genai import types

# Build conversation history
contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="What is 25 multiplied by 8?"
            )
        ],
    ),
    response.candidates[0].content,
]

# Create function response
function_response_kwargs = {
    "name": function_call.name,
    "response": {
        "result": result
    }
}

# Include function call ID if available
if getattr(function_call, "id", None):
    function_response_kwargs["id"] = function_call.id

function_response_part = types.Part.from_function_response(
    **function_response_kwargs
)

# Add tool result
contents.append(
    types.Content(
        role="user",
        parts=[function_response_part]
    )
)

# Ask Gemini for final answer
final_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents,
    config=types.GenerateContentConfig(
        tools=[tools]
    )
)

print("Final Response:")
print(final_response.text)

Final Response:
25 multiplied by 8 is 200.


# Task 3: Build a Raw Python Agent

The goal of this task is to implement a simple agent using only Python, the Gemini API, and a while loop. The agent will repeatedly inspect the model response, execute requested tools, return tool results to the model, and continue until a final answer is produced or the maximum number of iterations is reached.

In [41]:
from google.genai import types

def run_agent(user_request, max_iterations=5):
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=user_request)
            ]
        )
    ]

    iteration = 0

    while iteration < max_iterations:
        iteration += 1

        print(f"\n--- Iteration {iteration} ---")

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                tools=[tools]
            )
        )

        model_content = response.candidates[0].content
        function_call = None

        for part in model_content.parts:
            if part.function_call:
                function_call = part.function_call
                break

        if function_call is None:
            print("Agent finished.")
            print("Final Answer:", response.text)
            return response.text

        tool_name = function_call.name
        tool_args = dict(function_call.args)

        print("Tool:", tool_name)
        print("Arguments:", tool_args)

        if tool_name not in available_tools:
            raise ValueError(f"Unknown tool: {tool_name}")

        tool_result = available_tools[tool_name](**tool_args)

        print("Tool Result:", tool_result)

        contents.append(model_content)

        function_response_args = {
            "name": tool_name,
            "response": {
                "result": tool_result
            }
        }

        if getattr(function_call, "id", None):
            function_response_args["id"] = function_call.id

        function_response_part = types.Part.from_function_response(
            **function_response_args
        )

        contents.append(
            types.Content(
                role="user",
                parts=[function_response_part]
            )
        )

    print("Maximum iterations reached.")
    return None

In [ ]:
# Single Test 
result = run_agent(
    "What is 15 multiplied by 6?"
)

print("\nReturned Result:", result)


--- Iteration 1 ---
Tool: calculator
Arguments: {'operation': 'multiply', 'a': 15, 'b': 6}
Tool Result: 90

--- Iteration 2 ---
Agent finished.
Final Answer: 15 multiplied by 6 is 90.

Returned Result: 15 multiplied by 6 is 90.


In [43]:
result = run_agent(
    "First multiply 12 by 5. Then add 10 to the result. Give me the final answer."
)

print("\nReturned Result:", result)


--- Iteration 1 ---
Tool: calculator
Arguments: {'b': 5, 'operation': 'multiply', 'a': 12}
Tool Result: 60

--- Iteration 2 ---
Tool: calculator
Arguments: {'b': 10, 'operation': 'add', 'a': 60}
Tool Result: 70

--- Iteration 3 ---
Agent finished.
Final Answer: The final answer is 70.

Returned Result: The final answer is 70.


In [44]:
result = run_agent(
    "Calculate 25 multiplied by 4.",
    max_iterations=3
)

print("\nAgent Output:", result)


--- Iteration 1 ---
Tool: calculator
Arguments: {'operation': 'multiply', 'a': 25, 'b': 4}
Tool Result: 100

--- Iteration 2 ---
Agent finished.
Final Answer: 25 multiplied by 4 is 100.

Agent Output: 25 multiplied by 4 is 100.


## How the Raw Agent Works
The agent starts with the user's request and sends it to Gemini. If Gemini returns a function call, the Python program identifies the requested tool and its arguments, executes the tool, and adds the tool result back into the conversation. The while loop then sends the updated conversation to Gemini again.
The loop continues until Gemini produces a final natural-language response. A maximum iteration limit is included to prevent infinite loops and uncontrolled API usage.

## Task 3 Conclusion
A raw Python agent was successfully implemented using a while loop without an agent framework. The agent can inspect Gemini's function calls, execute tools, return observations, and continue working until a final answer is generated. The multi-step example demonstrated that the agent can perform multiple tool calls in sequence, while the maximum iteration limit provides a basic safety mechanism against infinite loops.

# Task 4: Memory & Logging
This task demonstrates the role of memory and logging in an agent system. Conversation memory preserves previous user and assistant interactions, while working memory stores temporary information needed to complete the current task. Logging records the agent's actions, tool calls, results, and decisions for debugging and monitoring.

In [46]:
conversation_memory = []

def add_message(role, content):
    conversation_memory.append({
        "role": role,
        "content": content
    })

def show_conversation_memory():
    print("Conversation Memory:")
    
    for message in conversation_memory:
        print(f"{message['role']}: {message['content']}")

In [47]:
add_message("user", "My name is Sami.")
add_message("assistant", "Nice to meet you, Sami.")
add_message("user", "I am learning AI agents.")

show_conversation_memory()

Conversation Memory:
user: My name is Sami.
assistant: Nice to meet you, Sami.
user: I am learning AI agents.


In [48]:
working_memory = {
    "current_task": None,
    "steps_completed": [],
    "tool_results": [],
    "iteration": 0
}

def reset_working_memory():
    working_memory["current_task"] = None
    working_memory["steps_completed"] = []
    working_memory["tool_results"] = []
    working_memory["iteration"] = 0

In [49]:
reset_working_memory()

working_memory["current_task"] = "Calculate 12 × 5 and then add 10"
working_memory["iteration"] = 1
working_memory["steps_completed"].append("12 × 5")
working_memory["tool_results"].append(60)

print("Working Memory:")
print(working_memory)

Working Memory:
{'current_task': 'Calculate 12 × 5 and then add 10', 'steps_completed': ['12 × 5'], 'tool_results': [60], 'iteration': 1}


In [50]:
# Logging
agent_logs = []

def log_event(event, details):
    agent_logs.append({
        "event": event,
        "details": details
    })

def show_logs():
    print("Agent Logs:")
    
    for i, log in enumerate(agent_logs, start=1):
        print(f"{i}. {log['event']}: {log['details']}")

In [53]:
# Test logging
agent_logs = []

log_event("START", "Agent received a new task")
log_event("TOOL_CALL", "calculator(a=12, b=5, operation='multiply')")
log_event("TOOL_RESULT", "60")
log_event("TOOL_CALL", "calculator(a=60, b=10, operation='add')")
log_event("TOOL_RESULT", "70")
log_event("FINISH", "Final answer generated")

show_logs()

Agent Logs:
1. START: Agent received a new task
2. TOOL_CALL: calculator(a=12, b=5, operation='multiply')
3. TOOL_RESULT: 60
4. TOOL_CALL: calculator(a=60, b=10, operation='add')
5. TOOL_RESULT: 70
6. FINISH: Final answer generated


In [58]:
working_memory = {
    "current_task": None,
    "steps_completed": [],
    "tool_results": [],
    "iteration": 0
}

agent_logs = []

def log_event(event, details):
    agent_logs.append({
        "event": event,
        "details": details
    })


def run_agent_with_memory(user_request, max_iterations=5):
    working_memory["current_task"] = user_request
    working_memory["steps_completed"] = []
    working_memory["tool_results"] = []
    working_memory["iteration"] = 0
    agent_logs.clear()

    log_event("START", user_request)

    messages = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_request)]
        )
    ]

    for iteration in range(max_iterations):

        working_memory["iteration"] = iteration + 1
        log_event("ITERATION", iteration + 1)

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config=types.GenerateContentConfig(
                tools=[tools]
            )
        )

        function_calls = []

        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_calls.append(part.function_call)

        if not function_calls:
            final_text = response.text

            log_event("FINISH", final_text)

            print("\nFINAL ANSWER:")
            print(final_text)

            return final_text

        messages.append(response.candidates[0].content)

        function_response_parts = []

        for function_call in function_calls:

            tool_name = function_call.name
            tool_args = dict(function_call.args)

            log_event(
                "TOOL_CALL",
                f"{tool_name}({tool_args})"
            )

            print(f"\nTOOL CALL: {tool_name}")
            print(f"INPUT: {tool_args}")

            try:
                if tool_name not in available_tools:
                    raise ValueError(f"Unknown tool: {tool_name}")

                result = available_tools[tool_name](**tool_args)

                working_memory["steps_completed"].append(
                    f"{tool_name} executed"
                )

                working_memory["tool_results"].append(result)

                log_event("TOOL_RESULT", result)

                print(f"OBSERVATION: {result}")

                function_response_parts.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response={
                            "result": result
                        }
                    )
                )

            except Exception as e:

                error_message = str(e)

                working_memory["tool_results"].append(
                    {
                        "error": error_message
                    }
                )

                log_event("TOOL_ERROR", error_message)

                print(f"TOOL ERROR: {error_message}")

                function_response_parts.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response={
                            "error": error_message
                        }
                    )
                )

        messages.append(
            types.Content(
                role="user",
                parts=function_response_parts
            )
        )

    log_event(
        "STOP",
        f"Maximum iterations ({max_iterations}) reached"
    )

    print("\nMAXIMUM ITERATIONS REACHED.")
    return None

In [59]:
result = run_agent_with_memory(
    "Calculate 12 multiplied by 5 and then add 10."
)


TOOL CALL: calculator
INPUT: {'operation': 'multiply', 'a': 12, 'b': 5}
OBSERVATION: 60

TOOL CALL: calculator
INPUT: {'operation': 'add', 'a': 60, 'b': 10}
OBSERVATION: 70

FINAL ANSWER:
12 multiplied by 5 is 60. Then, 60 plus 10 is 70.


In [60]:
print("WORKING MEMORY")
print(working_memory)

WORKING MEMORY
{'current_task': 'Calculate 12 multiplied by 5 and then add 10.', 'steps_completed': ['calculator executed', 'calculator executed'], 'tool_results': [60, 70], 'iteration': 3}


In [ ]:
# Actual Agent Log check
print("AGENT LOGS")

for i, log in enumerate(agent_logs, start=1):
    print(f"{i}. {log['event']}: {log['details']}")

AGENT LOGS
1. START: Calculate 12 multiplied by 5 and then add 10.
2. ITERATION: 1
3. TOOL_CALL: calculator({'operation': 'multiply', 'a': 12, 'b': 5})
4. TOOL_RESULT: 60
5. ITERATION: 2
6. TOOL_CALL: calculator({'operation': 'add', 'a': 60, 'b': 10})
7. TOOL_RESULT: 70
8. ITERATION: 3
9. FINISH: 12 multiplied by 5 is 60. Then, 60 plus 10 is 70.


## Conversation Memory vs Working Memory

Conversation memory stores the history of the interaction, such as previous user and assistant messages. It allows the agent to maintain context across multiple turns.
Working memory stores temporary information related to the current task. Examples include intermediate calculations, completed steps, tool results, and the current iteration number.
Conversation memory is useful for maintaining context, while working memory is useful for completing the current multi-step task.

## Task 4 Conclusion

Memory and logging are important components of reliable agent systems. Conversation memory preserves interaction history, while working memory stores temporary state and intermediate results during task execution. Logging provides visibility into the agent's actions and makes debugging and monitoring easier. Together, these mechanisms help agents maintain context, complete multi-step tasks, and diagnose failures.

# Task 5: Failure Modes & Guardrails

This task deliberately tests how an agent behaves when a tool cannot provide the requested information. The purpose is to observe failures and identify practical guardrails that make agent systems safer and more reliable.

In [62]:
failure_test = run_agent_with_memory(
    "Find the weather in Tokyo and tell me its temperature."
)


TOOL CALL: get_weather
INPUT: {'city': 'Tokyo'}
TOOL ERROR: Weather data is not available for Tokyo.

FINAL ANSWER:
I am sorry, but I cannot get the weather for Tokyo.


In [63]:
print("FAILURE TEST LOGS")

for i, log in enumerate(agent_logs, start=1):
    print(f"{i}. {log['event']}: {log['details']}")

FAILURE TEST LOGS
1. START: Find the weather in Tokyo and tell me its temperature.
2. ITERATION: 1
3. TOOL_CALL: get_weather({'city': 'Tokyo'})
4. TOOL_ERROR: Weather data is not available for Tokyo.
5. ITERATION: 2
6. FINISH: I am sorry, but I cannot get the weather for Tokyo.


## Failure Modes and Mitigations

| Failure Mode | What Can Happen | Mitigation |
|---|---|---|
| Infinite loops | Agent repeatedly calls tools. | Use max_iterations. |
| Wrong tool selection | Agent selects an unsuitable tool. | Use clear tool descriptions. |
| Wrong arguments | Tool receives invalid arguments. | Use JSON schemas and validation. |
| Tool errors | Tool fails during execution. | Use try/except error handling. |
| Unknown tool | Agent requests an unavailable tool. | Validate against the tool registry. |
| Hallucinated results | Agent invents information instead of using a tool. | Explicitly instruct the agent to use tools when required. |

## Observed Failure
The agent was given a request for weather information about Tokyo. Tokyo was not available in the simulated weather dataset, so the weather tool generated an error. The error was caught by the agent and returned as a tool error instead of crashing the program. This demonstrates the importance of exception handling and guardrails in agent systems.
## Why Do Agent Frameworks Exist?
The raw Python implementation shows that an agent consists of model calls, tool execution, observations, message history, state, and stopping conditions. Frameworks such as LangChain, LangGraph, and CrewAI provide reusable abstractions for these components. They become useful when agents require many tools, complex workflows, memory, retries, routing, state management, or multiple agents.
## Task 5 Conclusion
The failure test demonstrated that agents can encounter errors when tools cannot provide the requested information. Guardrails such as iteration limits, clear tool descriptions, JSON schemas, input validation, exception handling, and tool registries make agents more reliable. Testing failure cases is important because a robust agent should handle unexpected situations safely rather than only working when everything goes correctly.

In [64]:
# Task 5: Deliberate Failure Test

try:
    failure_test = run_agent_with_memory(
        "Find the weather in Tokyo and tell me its temperature."
    )
except Exception as e:
    print("ERROR HANDLED SUCCESSFULLY")
    print("Error:", str(e))


TOOL CALL: get_weather
INPUT: {'city': 'Tokyo'}
TOOL ERROR: Weather data is not available for Tokyo.
ERROR HANDLED SUCCESSFULLY
Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.342354904s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': '

In [65]:
print("FAILURE TEST LOGS")

for i, log in enumerate(agent_logs, start=1):
    print(f"{i}. {log['event']}: {log['details']}")

FAILURE TEST LOGS
1. START: Find the weather in Tokyo and tell me its temperature.
2. ITERATION: 1
3. TOOL_CALL: get_weather({'city': 'Tokyo'})
4. TOOL_ERROR: Weather data is not available for Tokyo.
5. ITERATION: 2


In [68]:
def run_agent_with_memory(user_request, max_iterations=5):
    working_memory["current_task"] = user_request
    working_memory["steps_completed"] = []
    working_memory["tool_results"] = []
    working_memory["iteration"] = 0
    agent_logs.clear()

    log_event("START", user_request)

    messages = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_request)]
        )
    ]

    for iteration in range(max_iterations):
        working_memory["iteration"] = iteration + 1
        log_event("ITERATION", iteration + 1)

        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=messages,
                config=types.GenerateContentConfig(
                    tools=[tools]
                )
            )
        except Exception as e:
            log_event("MODEL_ERROR", str(e))
            print("MODEL ERROR:", str(e))
            return None

        function_calls = [
            part.function_call
            for part in response.candidates[0].content.parts
            if part.function_call
        ]

        if not function_calls:
            final_text = response.text
            log_event("FINISH", final_text)
            print("\nFINAL ANSWER:")
            print(final_text)
            return final_text

        messages.append(response.candidates[0].content)
        function_response_parts = []

        for function_call in function_calls:
            tool_name = function_call.name
            tool_args = dict(function_call.args)

            log_event("TOOL_CALL", f"{tool_name}({tool_args})")
            print("\nTOOL CALL:", tool_name)
            print("INPUT:", tool_args)

            try:
                if tool_name not in available_tools:
                    raise ValueError(f"Unknown tool: {tool_name}")

                result = available_tools[tool_name](**tool_args)
                working_memory["steps_completed"].append(
                    f"{tool_name} executed"
                )
                working_memory["tool_results"].append(result)
                log_event("TOOL_RESULT", result)
                print("OBSERVATION:", result)

                function_response_parts.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response={"result": result}
                    )
                )

            except Exception as e:
                error_message = str(e)
                working_memory["tool_results"].append(
                    {"error": error_message}
                )
                log_event("TOOL_ERROR", error_message)
                print("TOOL ERROR:", error_message)

                function_response_parts.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response={"error": error_message}
                    )
                )

        messages.append(
            types.Content(
                role="user",
                parts=function_response_parts
            )
        )

    log_event("STOP", f"Maximum iterations ({max_iterations}) reached")
    print("\nMAXIMUM ITERATIONS REACHED.")
    return None

In [ ]:
# Final Test for Deliberate Failure
failure_test = run_agent_with_memory(
    "Find the weather in Tokyo and tell me its temperature."
)


TOOL CALL: get_weather
INPUT: {'city': 'Tokyo'}
TOOL ERROR: Weather data is not available for Tokyo.

FINAL ANSWER:
I am sorry, but I cannot retrieve weather information for Tokyo.
